# **Deep Learning and Neural Networks with Applications**


# **Week 9 Lab: AI Reasoning with LLMs**

This lab explores techniques for solving reasoning problems with large language models (LLMs). It consists of three tasks:
- **Task 1**: Use Zero-Shot Chain-of-Thought (CoT) prompting to solve a reasoning problem. This task includes two questions.
- **Task 2**: Use Self-Consistency (SC) prompting to solve a reasoning problem. This task includes two questions.
- **Task 3**: Implement a simple reasoning task using an LLM pre-trained for reasoning. This task includes two questions.

**Note for all tasks:** Depending on the problem, one reasoning technique may produce a more accurate result than another. This is especially true when we use small models, like we are using in this lab. Therefore, rather than focusing only on whether the final answer is correct, pay close attention to the reasoning process the model follows and identify where or how it may fail.

**Important:** Using a free GPU in Google Colab will speed up execution for all tasks in this lab. To enable a GPU, go to "Runtime", select "Change runtime type", choose one of the available GPUs under "Hardware accelerator" (for example, T4 GPU), and click "Save".

## **General Instructions**
1. Read the explanatory text and run each code cell in sequence.
2. Questions for each task appear throughout the notebook. Add your answers by following the specific instructions. Some questions require code, while others require analysis based on one or more outputs.
3. After completing all questions, submit your notebook through the corresponding QMPlus link for this lab assignment.

First, let's import the required libraries.

In [ ]:
from datasets import load_dataset #To load the GSM8K dataset, which is a collection of grade school math problems, for testing the model's reasoning capabilities.
import re #Regular expressions module for pattern matching and text manipulation.
from collections import Counter #Counter is a subclass of dict that helps count hashable objects. It is used to count the occurrences of elements in an iterable, such as a list or string, and returns a dictionary-like object where keys are the elements and values are their counts.

import torch #PyTorch library for tensor computations and deep learning.
from transformers import AutoModelForCausalLM, AutoTokenizer #Transformers library from Hugging Face, which provides pre-trained models and tokenizers for natural language processing tasks. AutoModelForCausalLM is used for causal language modeling, and AutoTokenizer is used to convert text into tokens that the model can understand.

In [ ]:
# Load dataset gsm8k
ds = load_dataset("openai/gsm8k", "main")

random_index = 0
sample = ds["test"][random_index] #and select a random question for our experiments

question = sample["question"]
answer = sample["answer"].split("####")
answer_path = answer[0]
final_answer = answer[1].strip()

print("Selected index:", random_index)
print("Question:", question)
print("Answer Path:", answer_path)
print("Final Answer:", final_answer)

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Selected index: 0
Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Answer Path: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.

Final Answer: 18


First, let's load a small model: Qwen2.5-0.5B-Instruct. We will use this model for Tasks 1 and 2.

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

Using device: cpu


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## Task 1: Zero-Shot Chain-of-Thought Prompting

In this task, we will answer the selected question below using Chain-of-Thought (CoT) prompting. CoT is an inference-time strategy that encourages the model to reason step by step before producing a final answer. In the original CoT approach, exemplars are provided to guide the model through this process. The original CoT paper is available here: https://proceedings.neurips.cc/paper/2022/hash/9d5609613524ecf4f15af0f7b31abca4-Abstract-Conference.html. Later, zero-shot CoT was proposed as an alternative. The zero-shot CoT paper is available here: https://proceedings.neurips.cc/paper_files/paper/2022/hash/8bb0d291acd4acf06ef112099c16f326-Abstract-Conference.html.

In [ ]:
#We will define a function for prompting our model using CoT (Chain of Thought) reasoning. The model will be instructed to first reason step-by-step and then provide the final answer.
def generate_cot_zeroshot(prompt: str, max_new_tokens: int = 250):
    messages = [
        {"role": "system", "content": "Solve the problem. First, write down your step-by-step reasoning. Finally, write the final answer clearly marked as 'Answer: <value>'."}, #Instructions for the model to reason step-by-step and provide the final answer
        {"role": "user", "content": prompt} #Our question is provided as a user message to the model
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) #Apply the chat template to format the messages for the model

    inputs = tokenizer(text, return_tensors="pt").to(device) #Tokenize the formatted text and move it to the appropriate device (GPU or CPU)

    with torch.no_grad(): #Disable gradient calculation for inference to save memory and computation
        outputs = model.generate(
            **inputs, #Pass the tokenized inputs to the model for generation
            max_new_tokens=max_new_tokens, #Limit the number of new tokens generated by the model
            do_sample=False, #Apply greedy decoding as in CoT
        )

    input_length = inputs.input_ids.shape[1] #Get the length of the input tokens to slice the output and retrieve only the generated part
    return tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True) #Decode the generated tokens into a string, skipping any special tokens used by the model

print("CoT Answer:\n", generate_cot_zeroshot(question, max_new_tokens=500)) #Print the CoT answer

CoT Answer:
 To determine how much Janet makes at the farmers' market each day, we need to follow these steps:

1. Calculate the total number of eggs laid by the ducks in a day.
2. Subtract the number of eggs eaten in the morning from the total number of eggs laid.
3. Subtract the number of eggs baked from the remaining eggs after breakfast.
4. Multiply the remaining eggs by the price at which they are sold.

Let's go through each step in detail:

1. **Calculate the total number of eggs laid by the ducks in a day:**
   - Janet's ducks lay 16 eggs per day.
   - She eats 3 eggs for breakfast every morning.
   - She bakes 4 muffins for her friends every day.
   - Therefore, the total number of eggs laid is \(16 + 3 + 4 = 23\).

2. **Subtract the number of eggs eaten in the morning from the total number of eggs laid:**
   - The number of eggs left after breakfast is \(23 - 3 = 20\).
   - After baking muffins, the number of eggs left is \(20 - 4 = 16\).

3. **Subtract the number of eggs bak

**Task 1 - Question 1:** What is the benefit of using an inference-time prompting technique such as Chain-of-Thought prompting, rather than fine-tuning a pre-trained model on labelled data for the specific task?

**Answer:**

The benefit of using an inference-time prompting tool like Chain-of-Thought (CoT) prompting instead of a fine tuned pre trained model is that CoT enhances the models reasoning without the need for more training. Lecture 9 defines CoT as a simple in-context learning method originally relying on two ideas. The first being giving exemplars to the model on how the output should be and the second being encouraging these exemplars to show a multi step process for reasoning. The lecture states the main benefit of these in context learning methods as a tool that focuses mechanisms to increase accuracy at inference time. Another reason why this technique is beneficial is because it doesn’t require as much data or as costs as a fine tuning pre trained models does. Lecture 9 (Evolution of AI Reasoning) states that fine tuning are used for good performance in task specific datasets (for example common sense reasoning and mathematical reasoning) however prompting techniques are the best solution to improve generated text at inference time without expensive fine tuning. Another reason is because fine tuning alone isn’t enough for reasoning, this is mentioned in lecture 9 (Early system 1 behaviour of LLMs) that models such as GPT-3 (even with fine tuning) would performance poorly in tasks requiring reasoning. Lastly, inference time promoting techniques such as CoT prompting can be more beneficial than fine tuning a pre trained model as it provides an inference time solutions to problems requiring reasoning steps rather than instant answers, which would perform well on labelled data for the specific task.

**Task 1 - Question 2:** What is the benefit of using zero-shot Chain-of-Thought prompting instead of the original Chain-of-Thought prompting approach (that is, providing a few exemplars of the multi-step reasoning process)?

**Answer:**

The benefit of using zero-shot chain of thought prompting instead of the original chain of thought prompting approach is that CoT doesn’t need multiple worked examples where as the original approach does, instead CoT adds a simple instructions such as “Lets think step by step”. Lecture 9 states that the original CoT prompting approach can be computationally expensive for three reasons which are increased number of input tokens, cost and latency of generating the output where higher and an extra cost of human time to engineer exemplars. Zero shot CoT promoting reduces these limitation as mentioned in the lecture LLMs are decent zero shot reasoners by adding “let’s think step by step” before each answer, which also saves input tokens.

## Task 2: Self-Consistency Prompting

Self-consistency prompting is another inference-time technique that guides a model to generate answers through a multi-step reasoning process. The paper is available here: https://arxiv.org/abs/2203.11171.

Although self-consistency prompting can also use exemplars to guide generation, in the next cell we will avoid using exemplars; that is, we will use zero-shot prompting.

In [ ]:
#We will build a function to wrap the model generation in a self-consistency framework. The model will be prompted multiple times to generate different reasoning paths, and we will collect the final answers from each path. The most common answer will be returned as the consensus answer.
def generate_self_consistency(prompt: str, num_paths: int = 5) -> str:
    messages = [
        {"role": "system", "content": "Solve the problem. Show your work step-by-step. End your response with 'Answer: <number>'."}, #Instructions for the model to reason step-by-step and provide the final answer
        {"role": "user", "content": prompt} #Our question is provided as a user message to the model
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) #Apply the chat template to format the messages for the model
    inputs = tokenizer(text, return_tensors="pt").to(device) #Tokenize the formatted text and move it to the appropriate device (GPU or CPU)

    answers = []

    batch_inputs = {k: v.repeat(num_paths, 1) for k, v in inputs.items()} #Batch the generations for parallel processing. We repeat the inputs 'num_paths' times

    with torch.no_grad(): #Disable gradient calculation for inference to save memory and computation
        outputs = model.generate(
            **batch_inputs, #Pass the tokenized inputs to the model for generation
            max_new_tokens=500, #Limit the number of new tokens generated by the model
            do_sample=True,        # Enable sampling
            temperature=1.5,       # Introduce variance
        )

    input_length = inputs.input_ids.shape[1] #Get the length of the input tokens to slice the output and retrieve only the generated part

    print("--- Sampled Paths ---")
    for i, seq in enumerate(outputs): #Iterate through each generated sequence
        decoded = tokenizer.decode(seq[input_length:], skip_special_tokens=True) #Decode the generated tokens into a string, skipping any special tokens used by the model
        print(f"Path {i+1}:\n{decoded}\n{'-'*30}") #Print the generated reasoning path for each sample

        # Simple extraction logic for the final numeric answer (looking for "Answer: X" or a trailing digit)
        match = re.search(r"Answer:\s*([\d\.]+)", decoded, re.IGNORECASE)
        if match:
            answers.append(match.group(1))
        else:
            # Fallback to the last sequence of digits found
            digits = re.findall(r"\d+", decoded)
            if digits:
                answers.append(digits[-1])

    #Majority Vote
    if not answers:
        return "No valid numeric answers found."

    vote_counter = Counter(answers) #Count the occurrences of each answer to determine the most common one
    most_common_answer, count = vote_counter.most_common(1)[0] #Get the most common answer and its count

    print(f"Votes collected: {dict(vote_counter)}") #Print the collected votes for transparency
    return f"Consensus Answer: {most_common_answer} (with {count}/{num_paths} votes)" #Return the most common answer along with the number of votes it received out of the total number of paths generated

print(generate_self_consistency(question, num_paths=5)) #Print the consensus answer obtained from the self-consistency framework

--- Sampled Paths ---
Path 1:
To solve Janet's problem, we need to break it down into smaller steps.

First, let's determine how many eggs each day Janet has left after accounting for breakfast and baking muffins.

- Janet's weekly total egg requirement is \(16 \times 7 = 112\) eggs.
- From today's perspective, she needs to account for breakfast consumption which amounts to 3 eggs (breakfast) and baking of muffins which adds an additional 4 eggs.
- Therefore, today's daily egg intake includes 7 total, plus the extra two from the next day, bringing to a new level of 9 eggs.

Given that she ate 3 breakfasts and baked muffins four days ago, her net increase in egg quantity due to these activities can be estimated by considering this:

\[ 3 \text{ (breakfast)} + 4 \text{ (muffins)} - 3 (baked muffins since yesterday) \]

However, we'll first calculate the number of eggs consumed on each day using our current understanding:

Today's consumption:
\[ 16 (eggs/day) - 7 (breakfast) - 4 (baked m

**Task 2 - Question 1:** What is the benefit of self-consistency prompting, as opposed to Chain-of-Thought (CoT) prompting, for reasoning problems?

**Anwser:**

The benefit of self consistency prompting rather than Chain of Though (CoT) prompting is that it generates diverse and multiple reasoning paths and chooses the most consistent answer in the final answer set. Lecture 9 (Self consistency prompting) states that the CoT  uses greedy decoding which forces the model to output a single reasoning path for an answer. This is not ideal as the lecture states it chooses the single most likely token and the higher probability may not be the optimal token globally at each time step. Lecture 9 states that self consistency prompting uses multiple and diverse reasoning paths where each reasoning path samples from a pool of likely tokens using ‘stochastic sampling’. The self consistency prompting selects the optimal answer by marginalising or voting. Another benefit of self consistency prompting is that it achieves crucial gains over CoT prompting in common sense, arithmetic and symbolic reasoning. Although the self consistency promoting is very beneficial for reasoning problems, the lecture also mentions that generating several reasoning paths substantially leads to latency and cost during inference.

**Task 2 - Question 2:** What is the role of the temperature parameter in self-consistency prompting? **Note:** You can experiment by changing this parameter in the code cell above to observe what happens.

**Answer:**

In self-consistency prompting, the temparature parameter role is to determines the degree of randomness in the token sample used by the model to generate every reasoning path. Lecture 9 (Greedy Decoding) states that self consistency uses stochastic sampling where yt is randomly sampled from the distribution of token probabilities. Also a temperature parameter T scares the degree of randomness. The lecture interprets temperate in stochastic sampling as when T>1 then distribution flattens toward a uniform distribution (causing more randomness), when 0 < T< 1 the distribution sharpens around the top logits (causing less randomness) and lastly when T=0 the distribution approaches applying argmax as in greedy decoding.

I conducted an experiment by changing the default temperature parameter (0.7) to a new parameter (1.5). When the parameter temperature ware 0.7 the consensus answer was 81, 1/5 votes  and the votes collected was 81, 18, 114, 54, 0. The results show reasoning in this parameter was more structured even though there was differences in the final answer, break down steps are very clear and easy to understand (total number of eggs, determine how many eaten, subtract the number of eggs, multiply by price per egg). On the other hand when the parameter temperature was changed to 1.5 the the answer were very different (2, 18, 64, 500, 12) and a consensus answer of 2 (with 1/5 votes). With this parameter change the reasoning is incoherent and the randomness increased. This is seen in the details such as ‘Janet drinks coffee’ and  ‘She leaves one piece of bacon’, this shows irrelevant details occurring. Also it generates unrealistic answers such as the number of eggs being 540 per day and the total Janet making at the farmers market each day being $1,500. Overall the results show that increasing the temperature parameter from 0.7 to 1.5 increased the sampled paths randomness. This also mentioned in lecture 9, when the T>1 (in this case 1.5) occurs the distribution flattens toward a uniform distribution (causing increased randomness), which was evident in this experiment.

## Task 3: Native Reasoning Model

In this final task, we will load and use a native reasoning model (that is, a System 2 reasoning model) to generate an answer to our question.

In [ ]:
native_model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

print(f"Loading {native_model_name}. This may take a few minutes...")
native_tokenizer = AutoTokenizer.from_pretrained(native_model_name)
native_model = AutoModelForCausalLM.from_pretrained(native_model_name).to(device)
print("DeepSeek-R1 loaded!")

Loading deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B. This may take a few minutes...


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.55GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

DeepSeek-R1 loaded!


In [ ]:
def run_native_reasoning(question):
    messages = [
        {"role": "system", "content": "Solve the problem. Show your work step-by-step. End your response with 'Answer: <number>'."}, #Instructions for the model to reason step-by-step and provide the final answer
        {"role": "user", "content": question} #Our question is provided as a user message to the model
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) #Apply the chat template to format the messages for the model
    inputs = native_tokenizer(text, return_tensors="pt").to(device) #DeepSeek-R1 prefers simple user prompts without restrictive system instructions

    print("Model is finding the answer (this may take a few minutes)...")
    with torch.no_grad(): #Disable gradient calculation for inference to save memory and computation
        outputs = native_model.generate(**inputs, max_new_tokens=500) #Generate the model's response to the input question, allowing for a maximum of 600 new tokens to be generated.

    full_output = native_tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True) #Decode the generated tokens into a string, skipping any special tokens used by the model

    print("\n\n Final answer: \n\n")
    print(full_output) #Print the final answer provided by the model, which is expected to be a concise response to the original question.

run_native_reasoning(question) # Run the question through the DeepSeek-R1 model for native reasoning


Model is finding the answer (this may take a few minutes)...


 Final answer: 


Okay, so Janet has these ducks laying eggs, and I need to figure out how much money she makes every day at the farmers' market. Let me start by breaking down the information given.

First, Janet's ducks lay 16 eggs per day. That's straightforward. So, total eggs per day = 16.

Next, she eats three eggs for breakfast every morning. So, breakfast consumption per day = 3. Since she eats breakfast every morning, I need to consider how many breakfasts she has in a day. But wait, in a single day, she has one breakfast at 7 AM, right? So, breakfast consumption per day = 3.

Then, she bakes muffins for her friends every day with four eggs. So, muffin baking per day = 4. That means she uses four eggs for baking muffins each day.

Now, she sells the remainder at the farmers' market daily for $2 per fresh duck egg. So, the selling price is $2 per egg, and she sells all the remaining eggs after accounting for breakfas

**Task 3 - Question 1:** What benefit do native reasoning models provide compared with using non-reasoning pre-trained models together with an inference-time technique such as CoT or self-consistency prompting?

**Answer:**

The benefit of a native reasoning model is that the ability to reason is already in the model itself as it learns in the pre training phase using reinforcement learning. During the lecture 9 we looked at native system 2 thinking in LLMs via RF and the lecture mentions several limitations in self consistency and CoT prompting. For example CoT focus on local best next token generation rather than global reasoning paths, Self consistency increases inference latency and cost and lastly ToT and GoT increase inference computational cost and time.

There are many benefits of of using native reasoning and one is that reasoning is already built into model and doesn’t add on methods. The lecture states that the solution for the limitations from CoT and self consistency prompting is to train for reasoning capabilities directly during pre training for an LLM, these are used in native reasoning model via reinforcement learning. Another benefit is that a reasoning model does the reasoning automatically and step by step itself, this is mentioned in the lecture (on slide AI Reasoning), reasoning models output an intermediate reasoning trace which is like an intermediate response that works through a task step by step. The lab task above uses a native reasoning model (DeepSeek-R1) and the results show the models step by step method first starting with 16 eggs, then subtracting 16 by 3, then subtracting 13 by 4 and lastly multiplying the result 9 x 2 making the total revenue $18. The model was careful and even double checked its self (‘wait, that seems straightforward’) without using inference time prompting’s models such as Self consistency and CoT.


**Task 3 - Question 2:** Should we use reasoning models in all cases, that is, for every possible problem? Why or why not?

**Answer:**

Reasoning models should not be used in all cases. Lecture 9 states that reasoning models are designed for and good at complex tasks for example advanced math problems, solving puzzles, common sense or logical reasoning and challenging coding tasks (ie system 2 problems). In system thinking LLM reasoning is improved using reinforcement learning and the goal is to develop reasoning capabilities without supervised data and on the models self evolution through pure RL process. On the other hand reasoning models are not used for simpler tasks like translation, knowledge based question answering or summarisation (ie system 1 problems). System 1 thinking models are fast and generate answers immediately. The reason why reasoning models shouldn’t be used on every problem is because they are expensive and inefficient. Although reasoning models perform well at complex tasks such as coding, logical puzzles and mathematics, they are costly, verbose and tend to overthink. Lecture 9 (Key Takeaways) states that standard LLMs are preferred for straightforward tasks such as summarisation and translations. The lecture also states that reasoning models are good at breaking down multi step problems, deductive reasoning and generalisation to novel problems, however they are bad at cheap and fast responses, simple tasks and knowledge based tasks. This is also evident in the task above where the model produced a long reasoning process at each step and even overthinking (‘Wait that seem straightforward, but let me make sure I haven’t missed anything’).
